In [ ]:
import pika
import time

credentials = pika.PlainCredentials('martin', 'martin00')
parameters =  pika.ConnectionParameters('149.62.71.186', credentials=credentials)
connection = pika.BlockingConnection(parameters)
channel = connection.channel()

channel.queue_declare(queue='task', durable=True)

def callback(ch, method, properties, body):
    time.sleep(int(body))
    print(f"Task {int(body)} completed!")
    #Ročno ack, ker ni nujno, da task opravimo!
    ch.basic_ack(delivery_tag = method.delivery_tag)

#don't dispatch a new message to a worker until it has processed and acknowledged the previous one
channel.basic_qos(prefetch_count=1)


channel.basic_consume(queue='task', 
                      auto_ack=False,
                      on_message_callback=callback)

print(' [*] Waiting for messages.')
channel.start_consuming()



### 1. Connection & Authentication
The first few lines set up the "handshake" with the RabbitMQ server.
* **Credentials:** Uses the username `martin` and password `martin00`.
* **Parameters:** Points to a specific IP address (`149.62.71.186`).
* **BlockingConnection:** This is a simple, synchronous way to keep the connection open while the script runs.

### 2. The "Durable" Queue
`channel.queue_declare(queue='task', durable=True)`
This ensures the `task` queue exists. The **`durable=True`** part is your insurance policy: it tells RabbitMQ to save the queue to the disk so that if the server restarts, your tasks aren't deleted.



### 3. The Work Logic (Callback)
The `callback` function is what happens when a message arrives:
* **Simulation:** `time.sleep(int(body))` treats the message content as a number of seconds. If you send "5", the worker "works" for 5 seconds.
* **Manual Ack:** `ch.basic_ack(...)` is the worker saying: *"Okay, I'm officially done with this specific task."* ### 4. Fair Dispatch (The "Smart" Part)
`channel.basic_qos(prefetch_count=1)`
This is arguably the most important line. By default, RabbitMQ just sends messages in a circle (Worker A, then B, then C). 
* **The Problem:** Worker A might get two "heavy" 10-second tasks, while Worker B gets two "light" 1-second tasks. Worker A gets overloaded while B sits idle.
* **The Solution:** `prefetch_count=1` tells RabbitMQ: **"Don't give me a new message until I've finished the one I'm currently holding."**

### 5. Reliable Consumption
`channel.basic_consume(..., auto_ack=False, ...)`
Because **`auto_ack` is False**, RabbitMQ won't delete the message the second it sends it. If your worker's power goes out halfway through a task, RabbitMQ will realize the "Ack" never came and will send that same task to a different worker.

### Summary
In short: This code creates a **smart, reliable worker** that won't take more than it can handle and won't lose data if it crashes.
